## LLM Pretraining

<div class="alert alert-block alert-success">
- This code demonstrates a toy example of how LLM pretraining works in a very simplified way.
- Your task is to complete the empty cells and fill the missing parts of the code indicated using the ellipsis "..."

Through this exercise you will learn:
- What a vocabulary construction looks like for a given dataset
- How tokenization can be done
- Encoding the data before training and decoding the data after generation
- Loss function most commonly used
- Making a forward pass
- Training elements like optimizers </div>

<div class="alert alert-block alert-warning">
Below demostrates a toy example that takes a text and used each character in the text as a "token". The code even if it works for you will probably not generate anything legible. The goal is that you should understand each element of the pretraining process. In reality, the training is a lot more sophisticated for many reasons - some being scale of the datasets, size of the models etc. The fundamentals on which these models are trained, however, can be demonstrated using this toy example. </div>

In [ ]:
!pip install torch

In [ ]:
import os
os.makedirs("pre-trained-model", exist_ok=True)

In [ ]:
# ─────────────────────────────────────────
# 1. Read the dataset
# ─────────────────────────────────────────
with open("../datasets/llm_pretraining.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(f"Total characters: {len(text)}")
print(f"First 100 chars:\n{text[:100]}")

In [ ]:
# ─────────────────────────────────────────
# 2. Build vocabulary
# ─────────────────────────────────────────
chars = sorted(list(set(text)))   # all unique characters
vocab_size = len(chars)

print(f"\nVocabulary ({vocab_size} chars): {''.join(chars)}")

In [ ]:
# ─────────────────────────────────────────
# 3. Tokenizer: character <-> integer
# ─────────────────────────────────────────
stoi = {ch: i for i, ch in enumerate(chars)}  # string → index
itos = {i: ch for i, ch in enumerate(chars)}  # index → string

In [ ]:
# ─────────────────────────────────────────
# Save tokenizer (stoi + itos mappings)
# In real LLMs this is the "tokenizer.json"
# ─────────────────────────────────────────
import json

tokenizer = {
    "stoi": stoi,   # char → int
    "itos": itos    # int  → char (keys must be strings for JSON)
}

with open("pre-trained-model/tokenizer.json", "w", encoding="utf-8") as f:
    json.dump(tokenizer, f, ensure_ascii=False, indent=2)

print("Tokenizer saved to tokenizer.json")

In [ ]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# quick test
print(f"\nEncode 'hi': {encode('hi')}")
print(f"Decode back:  {decode(encode('hi'))}")

In [ ]:
# ─────────────────────────────────────────
# 4. Encode full dataset + train/val split
# ─────────────────────────────────────────
import torch
import torch.nn as nn
from torch.nn import functional as F

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
print(f"\nDataset shape: {data.shape}, dtype: {data.dtype}")

n = int(0.9 * len(data))
train_data = data[:n]
val_data   = data[n:]        # BUG FIX: was data[:n] — now correctly uses the remaining 10%

In [ ]:
# ─────────────────────────────────────────
# 5. Batching
# Each input x predicts the next character y
# ─────────────────────────────────────────
batch_size = 4
block_size = 8   # context length: how many characters we look back

def get_batch(split):
    data_ = train_data if split == "train" else val_data
    ix = torch.randint(len(data_) - block_size, (batch_size,))   # random start positions
    x = torch.stack([data_[i:i+block_size]   for i in ix])       # input
    y = torch.stack([data_[i+1:i+block_size+1] for i in ix])     # target (shifted by 1)
    return x, y

xb, yb = get_batch("train")
print(f"\nInput batch shape:  {xb.shape}")
print(f"Target batch shape: {yb.shape}")

In [ ]:
# ─────────────────────────────────────────
# 6. Import Model Architecture
# ─────────────────────────────────────────

from model import BigramLanguageModel

In [ ]:
# ─────────────────────────────────────────
# 7. Test forward pass (before training)
# ─────────────────────────────────────────

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)

print(f"\nLogits shape: {logits.shape}")
print(f"Loss before training: {loss.item():.4f}")
print(f"Expected random loss: {torch.log(torch.tensor(vocab_size)).item():.4f}")  # -ln(1/vocab_size)

In [ ]:
# ─────────────────────────────────────────
# 8. Training loop
# ─────────────────────────────────────────
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32

for step in range(1000):
    xb, yb = get_batch("train")
    logits, loss = m(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(f"Step {step:4d} | loss: {loss.item():.4f}")

print(f"\nFinal loss: {loss.item():.4f}")

In [ ]:
# At the end of pretraining — save hyperparameters alongside weights
import json

config = {
    "model_type": "BigramLanguageModel",
    "vocab_size": vocab_size,
}

with open("pre-trained-model/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config saved to config.json")

torch.save(m.state_dict(), "pre-trained-model/bigram_pretrained.pt")
print("Model saved to bigram_pretrained.pt")

In [ ]:
# ─────────────────────────────────────────
# 9. Generate text from the trained model

# this step generates text using your trained model
# it probably won't generate anything interesting (or maybe it will?)
# by the time you reach this step you should have uncerstood the principals of:
# 1. How the pre-training works
# 2. Can go back to the corret sources to understand how to expand on this basic knowledge

# ─────────────────────────────────────────
input_data = torch.tensor([encode("Let us kill him")], dtype=torch.long)
generated   = m.generate(idx=input_data, max_new_tokens=500)

print("\n--- Generated text ---")
print(decode(generated[0].tolist()))